<table align="left">
  <td>
    <a href="https://colab.research.google.com/github/fabiobento/dnn-course-2026-1/blob/main/C1M4_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
  </td>
</table>

# Atividade Avaliativa - Superando o *Overfitting*: Construindo uma CNN Robusta

Bem-vindo à tarefa final deste curso! Você construiu uma base sólida no PyTorch, passando de tensores básicos para uma Rede Neural Convolucional completa e funcional em um laboratório anterior. Esse foi um primeiro passo essencial. Agora, é hora de dar o próximo passo e enfrentar um desafio que todo profissional de aprendizado profundo (*deep learning*) encontra: pegar um modelo promissor, mas com falhas, e elevá-lo de nível.

O seu modelo anterior mostrou sinais claros de *overfitting* (sobreajuste), um obstáculo comum onde a rede memoriza os dados de treinamento em vez de aprender a generalizar. Esta tarefa é a sua missão para resolver esse problema, não apenas ajustando um parâmetro, mas reengenharizando sistematicamente todo o seu *pipeline* de aprendizado de máquina com um conjunto de ferramentas e técnicas profissionais.

Para conseguir isso, você implementará uma estratégia multifacetada, atualizando cada componente da sua configuração:

* **Aprimorar o Pipeline de Dados** com um aumento de dados (*data augmentation*) mais poderoso para criar um conjunto de treinamento mais rico.
* **Refatorar a Arquitetura para Modularidade**, criando `CNNBlocks` reutilizáveis para um código mais limpo e escalável.
* **Integrar Camadas Avançadas** como a **Normalização em Lote** (*Batch Normalization*) para estabilizar o treinamento e melhorar a generalização.
* **Implementar uma Estratégia Robusta de Regularização** usando **Dropout** e **Decaimento de Pesos** (*Weight Decay*) para combater o *overfitting* diretamente.

Vamos começar e elevar o seu modelo para o próximo nível!

---
<a name='submission'></a>

<h4 style="color:green; font-weight:bold;">DICAS PARA O SUCESSO NA AVALIAÇÃO DA SUA TAREFA:</h4>

* Todas as células estão congeladas, exceto aquelas onde você precisa enviar suas soluções ou quando for explicitamente mencionado que você pode interagir com elas.

* Em cada célula de exercício, procure pelos comentários `### INICIE SEU CÓDIGO AQUI ###` e `### TERMINE SEU CÓDIGO AQUI ###`. Eles mostram onde você deve escrever o código da solução. **Não adicione nem altere nenhum código que esteja fora desses comentários**.

* Você pode adicionar novas células para fazer experimentos, mas elas serão ignoradas. Portanto, não dependa de células recém-criadas para hospedar o código da sua solução; utilize os locais fornecidos para isso.
---

## Índice
- [Importação de Bibliotecas](#0)
- [1 - Atualizando o seu Pipeline de Dados](#1)
    - [1.1 - Definindo Transformações Mais Poderosas](#1-1)
        - **[Exercício 1 - define_transformations](#ex-1)**
    - [1.2 - Montando os Data Loaders](#1-2)
    - [1.3 - Visualizando as Imagens de Treinamento](#1-3)
- [2 - Desenvolvenco uma CNN robusta e modular](#2)
    - [2.1 - O poder da Modularidade: O CNNBlock](#2-1)
        - [2.1.1 - Camada BatchNorm2d](#2-1-1)
            - **[Exercício 2 - CNNBlock](#ex-2)**
    - [2.2 - Montando a CNN Completa com Blocos Modulares](#2-2)
        - **[Exercício 3 - SimpleCNN](#ex-3)**
- [3 - Treinando o Modelo Atualizado](#3)
    - [3.1 - Configurando a Perda (Loss) e o Otimizador](#3-1)
    - [3.2 - Implementando a Lógica de Treinamento e Validação](#3-2)    
        - **[Exercício 4 - train_epoch](#ex-4)**
        - **[Exercício 5 - validate_epoch](#ex-5)**        
- [4 - Além dos Fundamentos: Uma visão próximo nível](#4)

<a name='0'></a>

## Importação de Bibliotecas

In [ ]:
# Importa o módulo copy para permitir a criação de cópias rasas (shallow) ou profundas (deep) de objetos
import copy 

# Importa a biblioteca principal do PyTorch, utilizada para computação baseada em tensores e deep learning
import torch

# Importa o módulo de redes neurais (Neural Networks), que contém camadas, funções de ativação e perdas
import torch.nn as nn

# Importa o módulo de otimização, que inclui algoritmos como SGD, Adam e RMSprop
import torch.optim as optim

# Importa as transformações do torchvision para processamento, redimensionamento e aumento de dados de imagem
import torchvision.transforms as transforms

# Importa o DataLoader para gerenciar a divisão dos dados em lotes (batches), embaralhamento e paralelismo
from torch.utils.data import DataLoader

In [ ]:
# Importa o script ou módulo local 'helper_utils', que contém funções utilitárias personalizadas
# para o projeto (como funções de carregamento de dados, plotagem de métricas ou visualizações)
import helper_utils

# Importa o módulo ou script local 'unittests', que contém os testes automatizados
# estruturados para validar o comportamento e a integridade das funções e modelos desenvolvidos
import unittests

In [ ]:
# Configuração do dispositivo de hardware (device)
# Define o uso de GPU ('cuda') se uma placa de vídeo compatível estiver disponível; caso contrário, usa a CPU ('cpu')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Exibe no console qual dispositivo de hardware foi selecionado para rodar os cálculos
print(f"Dispositivo em uso (Using device): {device}")

<a name='1'></a>
## 1 - Atualizando o seu Pipeline de Dados

No primeiro laboratório deste módulo, você construiu um classificador CNN poderoso do zero. Embora tenha funcionado, você também encontrou um obstáculo clássico do aprendizado de máquina: o ***overfitting*** (sobreajuste). O seu modelo começou a memorizar os dados de treinamento em vez de aprender a generalizar, um problema comum quando o desempenho de um modelo nos dados de validação estagna ou se degrada.

Os seus resultados de treinamento daquele laboratório provavelmente produziram um gráfico semelhante ao abaixo. Ele ilustra perfeitamente este desafio, mostrando os sinais reveladores de *overfitting*. Observe atentamente a crescente lacuna entre a **perda de treinamento** (*training loss*), que continua a melhorar, e a **perda de validação** (*validation loss*), que estagna ou até mesmo piora. Essa divergência, juntamente com a **precisão de validação** (*validation accuracy*) atingindo um platô, é a evidência clássica de um modelo que está memorizando os dados de treinamento em vez de realmente aprender a generalizar.

![Lab 1 Gráfico de Treino](https://raw.githubusercontent.com/fabiobento/dnn-course-2026-1/main/nb_image/lab_1_training_plot.png)

Você agora enfrentará esse desafio de frente. O objetivo é duplo: primeiro, resolver o *overfitting* e, segundo, elevar o desempenho do seu modelo a novos patamares. Antes que você possa aprimorar a arquitetura do modelo, no entanto, você deve primeiro aprimorar os dados com os quais ele aprende.

Uma estratégia fundamental para construir modelos mais robustos é o **aumento de dados** (*data augmentation*). Ao criar versões modificadas das suas imagens de treinamento — espelhando-as, girando-as —, você ensina o seu modelo a reconhecer os elementos em uma variedade de condições. Essa técnica é uma primeira linha de defesa fundamental contra o *overfitting*. A sua primeira tarefa é construir um conjunto ainda mais poderoso de transformações de imagem para turbinar o seu conjunto de dados.

<a name='1-1'></a>
### 1.1 - Definindo Transformações Mais Poderosas

Vamos começar configurando os componentes essenciais para o seu *pipeline* de dados. Você começará definindo os valores padrão de normalização para o conjunto de dados CIFAR-100 e, em seguida, criará os próprios *pipelines* de transformação.

* Defina o `cifar100_mean` e o `cifar100_std`, os valores de média e desvio padrão para o conjunto de dados **CIFAR-100**.
* Define the `cifar100_mean` and `cifar100_std`, the mean and standard deviation values for the **CIFAR-100** dataset.

In [ ]:
# Média pré-calculada para cada um dos 3 canais (RGB) do conjunto de dados CIFAR-100
cifar100_mean = (0.5071, 0.4867, 0.4408)

# Desvio padrão pré-calculado para cada um dos 3 canais (RGB) do conjunto de dados CIFAR-100
cifar100_std = (0.2675, 0.2565, 0.2761)

Como você aprendeu anteriormente, o *pipeline* de transformação de treinamento é onde você aplica o aumento de dados (*data augmentation*). Para tornar o seu modelo ainda mais robusto, desta vez você adicionará uma nova técnica ao seu arsenal: `RandomVerticalFlip`. Embora o espelhamento horizontal seja comum, adicionar espelhamentos verticais também pode ajudar o modelo a aprender que a orientação de um objeto pode não estar sempre na posição vertical, um recurso útil para classificar coisas como insetos ou flores de vários ângulos.

<a name='ex-1'></a>
### Exercício 1 - define_transformations

Sua tarefa é definir dois *pipelines* distintos de transformação de imagens usando o `torchvision.transforms`.

**Sua Tarefa**:

* **Para `train_transformations`**: Crie uma composição de transformações para o conjunto de dados de treinamento.
    * Este *pipeline* deve incluir espelhamentos aleatórios na [horizontal](https://docs.pytorch.org/vision/main/generated/torchvision.transforms.RandomHorizontalFlip.html) e na [vertical](https://docs.pytorch.org/vision/main/generated/torchvision.transforms.RandomVerticalFlip.html).
    * Ele também deve [girar](https://docs.pytorch.org/vision/main/generated/torchvision.transforms.RandomRotation.html) as imagens aleatoriamente em até **15 graus**.
    * Por fim, ele deve converter as imagens para [tensores](https://docs.pytorch.org/vision/main/generated/torchvision.transforms.ToTensor.html) do PyTorch e [normalizá-las](https://docs.pytorch.org/vision/main/generated/torchvision.transforms.Normalize.html) usando as variáveis `mean` e `std` fornecidas.
* **Para `val_transformations`**: Crie um segundo *pipeline*, mais simples, para o conjunto de dados de validação.
    * Este *pipeline* deve executar apenas os dois passos essenciais: converter as imagens para [tensores](https://docs.pytorch.org/vision/main/generated/torchvision.transforms.ToTensor.html) e [normalizá-las](https://docs.pytorch.org/vision/main/generated/torchvision.transforms.Normalize.html) com as mesmas variáveis `mean` e `std`.

<details>
<summary><b><font color="green">Dicas de Código Adicionais (Clique para expandir se estiver travado)</font></b></summary>

Se você estiver com dificuldades, aqui está uma explicação mais detalhada.

Você usará `transforms.Compose([...])` para criar uma lista de transformações para ambos os *pipelines*. Todas as funções necessárias fazem parte do módulo `transforms`.

**Para `train_transformations`**:

* Você precisa criar uma lista de cinco objetos de transformação dentro do `transforms.Compose`.
* O primeiro é para espelhamentos horizontais. A chamada se parece com isto: `transforms.RandomHorizontalFlip()`.
* Os dois próximos, para espelhamentos verticais e rotações, seguem um padrão semelhante. Lembre-se de passar `15` como o argumento para a rotação.
* As duas últimas transformações são:
    * `chame o método ToTensor do módulo transforms`
    * `chame o método Normalize do módulo transforms, passando para ele as variáveis mean e std`



**Para `val_transformations`**:

* Este *pipeline* é muito mais simples e contém apenas as duas últimas etapas do *pipeline* de treinamento.
* A sua lista dentro do `transforms.Compose` deve conter apenas dois itens:
    * `primeiro, a transformação para converter uma imagem em um tensor`
    * `segundo, a transformação para normalizar o tensor usando as variáveis mean e std fornecidas`

</details>

In [ ]:
# FUNÇÃO AVALIADA: define_transformations

def define_transformations(mean, std):
    """
    Cria pipelines de transformações de imagem para treinamento e validação.

    Argumentos:
        mean (lista ou tupla): Uma sequência de valores de média para cada canal.
        std (lista ou tupla): Uma sequência de valores de desvio padrão para cada canal.

    Retorna:
        train_transformations (torchvision.transforms.Compose): O pipeline de 
                                                                transformações de treinamento.
        val_transformations (torchvision.transforms.Compose): O pipeline de 
                                                              transformações de validação.
    """
    
    ### INICIE O SEU CÓDIGO AQUI ###
    
    # Define a sequência de transformações para o conjunto de dados de treinamento.
    
    train_transformations = None.None([
        # Inverte a imagem horizontalmente de forma aleatória com uma probabilidade de 50%.
        None,
        # Inverte a imagem verticalmente de forma aleatória com uma probabilidade de 50%.
        None,
        # Rotaciona a imagem por um ângulo aleatório entre -15 e +15 graus.
        None,
        # Converte a imagem de uma Imagem PIL ou array NumPy para um tensor do PyTorch.
        None,
        # Normaliza a imagem em formato de tensor com a média e o desvio padrão fornecidos.
        None
    ]) 
    
    # Define a sequência de transformações para o conjunto de dados de validação.
    val_transformations = None.None([
        # Converte a imagem de uma Imagem PIL ou array NumPy para um tensor do PyTorch.
        None,
        # Normaliza a imagem em formato de tensor com a média e o desvio padrão fornecidos.
        None
    ]) 
    
    ### TÉRMINE O SEU CÓDIGO AQUI ###

    # Retorna ambos os pipelines de transformação.
    return train_transformations, val_transformations

In [ ]:
# Verifica as Transformações
print("--- Verificando define_transformations ---\n")
train_transform_verify, val_transform_verify = define_transformations(cifar100_mean, cifar100_std)


print("Transformações de Treinamento:")
print(train_transform_verify)
print("-" * 30)
print("\nTransformações de Validação:")
print(val_transform_verify)

#### Saída Esperada:

```
Transformações de Treinamento:
Compose(
    RandomHorizontalFlip(p=0.5)
    RandomVerticalFlip(p=0.5)
    RandomRotation(degrees=[-15.0, 15.0], interpolation=nearest, expand=False, fill=0)
    ToTensor()
    Normalize(mean=(0.5071, 0.4867, 0.4408), std=(0.2675, 0.2565, 0.2761))
)
------------------------------

Transformações de Validação:
Compose(
    ToTensor()
    Normalize(mean=(0.5071, 0.4867, 0.4408), std=(0.2675, 0.2565, 0.2761))
)
```

* Chame a função `define_transformations`, passando `cifar100_mean` e `cifar100_std` como argumentos.
* Isso retorna dois *pipelines* de transformação separados, que são armazenados nas variáveis `train_transform` e `val_transform` para uso posterior.

In [ ]:
# Cria e armazena os pipelines de transformação para treinamento e validação
train_transform, val_transform = define_transformations(cifar100_mean, cifar100_std)

<a name='1-2'></a>
### 1.2 - Montando os Data Loaders

Com os seus novos e poderosos *pipelines* de transformação definidos, é hora de preparar os dados para o treinamento. Você primeiro especificará as 15 classes alvo e, em seguida, usará as suas transformações para carregar as imagens e empacotá-las em objetos `DataLoader`, que alimentarão o seu modelo com dados em lotes (*batches*).

* Primeiro, defina a lista `all_target_classes`.
* Estas são as mesmas classes de flores, mamíferos e insetos com as quais você trabalhou no laboratório anterior, garantindo que você esteja lidando com o mesmo problema de classificação, mas com um *pipeline* atualizado.

In [ ]:
# Define a lista completa de classes que serão utilizadas como alvo (target).
all_target_classes = [
    # Flores
    'orchid', 'poppy', 'rose', 'sunflower', 'tulip',
    # Mamíferos
    'fox', 'porcupine', 'possum', 'raccoon', 'skunk',
    # Insetos
    'bee', 'beetle', 'butterfly', 'caterpillar', 'cockroach'
]

* Em seguida, chame a função `load_cifar100_subset`, passando a sua lista de classes (`all_target_classes`) e ambos os *pipelines* de transformação (`train_transform` e `val_transform`).
* Esta função lida com todo o processo de carregamento e retorna dois objetos `Dataset` do PyTorch, que são armazenados nas variáveis `train_dataset` e `val_dataset`.

In [ ]:
# Carrega os conjuntos de dados (datasets) completos baseados no subconjunto selecionado.
train_dataset, val_dataset = helper_utils.load_cifar100_subset(all_target_classes, train_transform, val_transform)

<br>

Com os seus conjuntos de dados preparados, o passo final é empacotá-los no `DataLoader` do PyTorch. Este utilitário é essencial para alimentar o seu modelo com dados em lotes gerenciáveis (*batches*).

* Crie o `train_loader` para os seus dados de treinamento.
* Crie o `val_loader` para os seus dados de validação.

In [ ]:
# Define o número de amostras que serão processadas em cada lote (batch)
batch_size = 64

# Cria um carregador de dados (data loader) para o conjunto de treinamento, com o embaralhamento ativado (shuffle=True)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

# Cria um carregador de dados (data loader) para o conjunto de validação, sem embaralhamento (shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

<a name='1-3'></a>
### 1.3 - Visualizando as Imagens de Treinamento

É sempre uma boa prática visualizar os seus dados. A linha a seguir chama uma função auxiliar para exibir uma grade de imagens aleatórias do seu `train_loader`.

Preste muita atenção à saída. Como essas imagens vêm do conjunto de treinamento, você deve ver os efeitos do seu *pipeline* de aumento de dados (*data augmentation*) em ação. Procure por imagens que foram espelhadas horizontal ou verticalmente, ou giradas de forma aleatória. Esta é uma excelente maneira de confirmar se as suas transformações estão funcionando como o esperado.

In [ ]:
# Visualiza uma grade (grid) com imagens aleatórias do conjunto de treinamento
helper_utils.visualise_images(train_loader, grid=(3, 5))

<a name='2'></a>
## 2 - Construindo uma CNN Modular e Robusta

Com um *pipeline* de dados mais robusto em vigor, o seu próximo passo é aprimorar a própria arquitetura do modelo. Você irá refatorar a CNN original para ser mais modular, eficiente e poderosa. Este é o próximo passo fundamental para resolver o problema de *overfitting* e elevar o desempenho do seu modelo a novos patamares.

<a name='2-1'></a>
### 2.1 - O Poder da Modularidade: O CNNBlock

No laboratório anterior, a arquitetura do seu modelo tinha um padrão repetitivo de camadas de convolução, ativação e *pooling*. Definir essas camadas individualmente pode se tornar repetitivo e torna o modelo mais difícil de modificar. Uma abordagem muito melhor é agrupar esses padrões em um único módulo reutilizável. A sua primeira tarefa é criar um `CNNBlock` que empacote essas camadas juntas. Esse *design* modular torna o código do seu modelo principal significativamente mais limpo e fácil de gerenciar.

<a name='2-1-1'></a>
#### 2.1.1 - A Camada `BatchNorm2d`

Como parte deste novo bloco aprimorado, você também introduzirá uma nova e poderosa camada: `BatchNorm2d`. Essa camada é uma técnica fundamental para construir redes neurais profundas modernas e de alto desempenho.

Pense na Normalização em Lote (*Batch Normalization*) como um controlador de tráfego para os dados que fluem entre as camadas da sua rede. Depois que uma camada convolucional processa um lote (*batch*) de imagens, as saídas (ou ativações) podem ter distribuições que variam muito de um lote para o outro. A `BatchNorm2d` entra em ação e normaliza essas ativações dentro de cada minilote (*mini-batch*), ajustando-as para ter uma média e um desvio padrão consistentes. Em seguida, ela usa dois parâmetros que podem ser aprendidos (*learnable parameters*) para dimensionar e deslocar essa saída normalizada, permitindo que a própria rede aprenda a distribuição ideal para os dados naquele ponto.

Esse passo aparentemente simples oferece três benefícios profundos:

* **Estabiliza e Acelera o Treinamento:** Ao manter a distribuição dos dados consistente entre as camadas, evita-se que as camadas posteriores tenham de se adaptar constantemente a uma entrada instável proveniente das camadas anteriores. Essa estabilidade permite que você use taxas de aprendizado (*learning rates*) mais altas, o que pode acelerar drasticamente a rapidez com que o seu modelo aprende.
* **Atua como um Regularizador:** Como as estatísticas de normalização são calculadas para cada minilote exclusivo, é introduzida uma pequena quantidade de ruído no processo de treinamento. Esse ruído torna mais difícil para o modelo memorizar perfeitamente os dados de treinamento, incentivando-o a aprender recursos mais gerais e, assim, reduzindo o *overfitting*.
* **Reduz a Sensibilidade à Inicialização:** A camada torna o seu modelo menos dependente dos pesos aleatórios específicos com os quais ele começa, levando a resultados de treinamento mais confiáveis e fáceis de se repetir.

Ao adicionar `BatchNorm2d` ao seu `CNNBlock`, você não está apenas adicionando outra camada; você está, fundamentalmente, tornando o processo de treinamento do seu modelo mais estável, eficiente e robusto.

<a name='ex-2'></a>
### Exercício 2 - CNNBlock

Você implementará agora a classe `CNNBlock`. Esta classe irá empacotar as quatro camadas em um único módulo `nn.Sequential`.

**Sua Tarefa**:

**Dentro do método `__init__**`:

> * Você precisa definir um contêiner sequencial chamado `self.block`.
> * Dentro deste contêiner [nn.Sequential](https://docs.pytorch.org/docs/stable/generated/torch.nn.Sequential.html), você adicionará as seguintes camadas, em ordem:
> 1. Uma camada [nn.Conv2d](https://docs.pytorch.org/docs/stable/generated/torch.nn.Conv2d.html). Use os argumentos `in_channels`, `out_channels`, `kernel_size` e `padding` que são passados para o método `__init__`.
> 2. Uma camada [nn.BatchNorm2d](https://docs.pytorch.org/docs/stable/generated/torch.nn.BatchNorm2d.html). Esta camada precisa saber o número de canais da sua entrada, que é a saída da camada convolucional anterior.
> 3. Uma função de ativação [nn.ReLU](https://docs.pytorch.org/docs/stable/generated/torch.nn.ReLU.html).
> 4. Uma camada [nn.MaxPool2d](https://docs.pytorch.org/docs/stable/generated/torch.nn.MaxPool2d.html). Isso fará a redução de dimensionalidade (*downsampling*) do mapa de características. Você deve definir tanto o `kernel_size` quanto o `stride` como `2`.
> 

**Dentro do método `forward**`:

> * Este método executa a passagem direta (*forward pass*).
> * Passe o tensor de entrada `x` através do `self.block` que você definiu e retorne o resultado.

>

<details>
<summary><b><font color="green">Dicas Adicionais de Código (Clique para expandir se estiver com dificuldades)</font></b></summary>

Se você estiver procurando por mais orientação, aqui está uma explicação mais detalhada.

**Para o método `__init__**`:

* Você está definindo uma sequência de camadas. A sequência inteira será atribuída a `self.block`. A estrutura começa assim: `self.block = nn.Sequential(...)`.
* As camadas são fornecidas como argumentos para o `nn.Sequential`, separadas por vírgulas.
* **1. Camada Convolucional**: A primeira camada é `nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, padding=padding)`. Observe como ela usa os parâmetros da assinatura do método `__init__`.
* **2. Camada Batch Norm**: A segunda camada é `nn.BatchNorm2d(...)`. Ela precisa de um argumento: o número de canais que irá normalizar. Isso é igual ao número de canais de saída da camada anterior, que é `out_channels`.
* **3. Camada ReLU**: A terceira camada é simplesmente `nn.ReLU()`. Ela não requer nenhum argumento.
* **4. Camada de Max Pooling**: A camada final é `nn.MaxPool2d(...)`. Você precisa fornecer o `kernel_size` e o `stride`. A chamada ficará assim: `nn.MaxPool2d(kernel_size=2, stride=2)`.

**Para o método `forward**`:

* Esta é uma única linha de código. Você simplesmente precisa aplicar o módulo que você criou no método `__init__` ao tensor de entrada.
* O pseudocódigo seria: `retorne o resultado de aplicar self.block à entrada x`.

</details>

In [ ]:
# CLASSE AVALIADA: CNNBlock

class CNNBlock(nn.Module):
    """
    Define um bloco convolucional único para uma CNN.

    Este bloco consiste em uma camada convolucional, normalização em lote (Batch Normalization),
    uma ativação ReLU e uma camada de max-pooling, encapsuladas como um módulo sequencial.
    """
    def __init__(self, in_channels, out_channels, kernel_size=3, padding=1):
        """
        Inicializa as camadas do CNNBlock.

        Argumentos:
            in_channels (int): Número de canais na imagem de entrada.
            out_channels (int): Número de canais produzidos pela convolução.
            kernel_size (int, opcional): Tamanho do kernel convolucional. Padrão é 3.
            padding (int, opcional): Preenchimento com zeros (zero-padding) adicionado a ambos os lados da entrada. Padrão é 1.
        """
        # Inicializa a classe pai nn.Module.
        super(CNNBlock, self).__init__()
        
        ### INICIE O SEU CÓDIGO AQUI ###
        
        # Define o contêiner sequencial para as camadas do bloco.
        self.block = None(
            # Camada convolucional 2D para aplicar filtros aprendíveis à entrada.
            None,
            # Normalização em lote para estabilizar e acelerar o treinamento.
            None,
            # Função de ativação ReLU para introduzir não-linearidade.
            None,
            # Camada de max pooling para subamostrar o mapa de características e reduzir as dimensões espaciais.
            None
        ) 
        
        ### TÉRMINE O SEU CÓDIGO AQUI ###

    def forward(self, x):
        """
        Define a propagação para frente (forward pass) do CNNBlock.

        Argumentos:
            x: O tensor de entrada para o bloco.

        Retorna:
            O tensor de saída após passar pelas camadas do bloco.
        """
        
        ### INICIE O SEU CÓDIGO AQUI ###
        
        # Passa o tensor de entrada através do bloco sequencial de camadas.
        return None
    
        ### TÉRMINE O SEU CÓDIGO AQUI ###

In [ ]:
# Verifica o CNNBlock
print("--- Verificando CNNBlock ---\n")

# Instancia o bloco com 3 canais de entrada (ex: imagens RGB) e 16 canais de saída
verify_cnn_block = CNNBlock(in_channels=3, out_channels=16)
print("Estrutura do Bloco (Block Structure):\n")
print(verify_cnn_block)

# Verifica o formato (shape) da saída após uma propagação para frente (forward pass)
# Cria um tensor de entrada fictício (tamanho_do_lote=1, canais=3, altura=32, largura=32)
dummy_input = torch.randn(1, 3, 32, 32)
print(f"\nFormato do tensor de entrada: {dummy_input.shape}")

# Passa o tensor fictício através do bloco convolucional criado
output = verify_cnn_block(dummy_input)
print(f"Formato do tensor de saída:   {output.shape}")

#### Saída Esperada:

```
Block Structure:

CNNBlock(
  (block): Sequential(
    (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
)

Formato do tensor de entrada:  torch.Size([1, 3, 32, 32])
Formato do tensor de saída: torch.Size([1, 16, 16, 16])
```

In [ ]:
# Teste o seu código! (Executa os testes unitários automatizados para o CNNBlock)
unittests.exercise_2(CNNBlock)

<a name='2-2'></a>
### 2.2 - Montando a CNN Completa com Blocos Modulares

Agora que você tem um `CNNBlock` reutilizável, você pode montar a sua arquitetura `SimpleCNN` completa. Ao usar o seu novo bloco modular, você verá como a definição do seu modelo se torna muito mais limpa e profissional. Em vez de definir muitas camadas individuais para a parte convolucional da sua rede, agora você definirá apenas três instâncias de `CNNBlock`.

O seu modelo consistirá em duas partes principais:

* **Um extrator de características (*feature extractor*)**: Uma sequência de três `CNNBlocks` que aprenderão a identificar padrões visuais nas imagens.
* **Um classificador**: Uma sequência de camadas totalmente conectadas (*fully connected layers*) que pegará as características extraídas pelos blocos convolucionais e fará a previsão final.

Nesta nova versão, você também aumentará a **taxa de *dropout* para `0.6**`. Este é outro passo importante na sua luta contra o *overfitting*, pois torna o modelo menos propenso a depender de qualquer característica única isolada.

<a name='ex-3'></a>
### Exercício 3 - SimpleCNN

Você implementará agora os métodos `__init__` e `forward` para a classe `SimpleCNN`. Você usará o `CNNBlock` que acabou de construir como o componente principal do corpo da rede.

**Sua Tarefa**:

**Dentro do método `__init__`**:

> * **Extrator de Características (*Feature Extractor*)**:
> * Instancie três camadas `CNNBlock` (`conv_block1`, `conv_block2`, `conv_block3`).
> * O primeiro bloco deve receber uma entrada com **3 canais** (para imagens RGB) e produzir **32 canais de saída**.
> * Para os blocos subsequentes, o número de canais de entrada deve corresponder ao número de canais de saída do bloco anterior. Você dobrará o número de canais a cada passo `(3 -> 32 -> 64 -> 128)`.
> 
> 

> * **Classificador**:
> * Defina um `self.classifier` usando um contêiner `nn.Sequential`.
> * Este contêiner deve ter as seguintes camadas, em ordem:
> 1. Uma camada [nn.Flatten](https://docs.pytorch.org/docs/stable/generated/torch.nn.Flatten.html) para transformar o mapa de características 2D em um vetor 1D.
> 2. Uma camada [nn.Linear](https://docs.pytorch.org/docs/stable/generated/torch.nn.Linear.html). Você deve calcular o número correto de características de entrada. Isso depende do formato de saída do último `CNNBlock`. O tamanho de saída desta camada deve ser **512**.
> 3. Uma ativação `nn.ReLU`.
> 4. Uma camada [nn.Dropout](https://docs.pytorch.org/docs/stable/generated/torch.nn.Dropout.html) com uma taxa de `0.6` para ajudar a evitar o *overfitting*.
> 5. Uma camada `nn.Linear` final que mapeia as **512 características** para o **número de classes de saída**.
> 

**Dentro do método `forward`**:

> * Defina o fluxo de dados através da rede.
> * Passe a entrada `x` sequencialmente por `conv_block1`, depois `conv_block2` e, em seguida, `conv_block3`.
> * Por fim, passe a saída do último bloco convolucional através do seu `classifier`.
> * Retorne a saída final.
> 
>

<details>
<summary><b><font color="green">Dicas Adicionais de Código (Clique para expandir se estiver com dificuldades)</font></b></summary>

Se você estiver com dificuldades, aqui está uma explicação mais detalhada para a implementação.

**Para o método `__init__**`:

* **Blocos Convolucionais**:

```
* O primeiro bloco é uma instanciação direta: `self.conv_block1 = CNNBlock(in_channels=3, out_channels=32)`.
* Para o segundo bloco, o `in_channels` deve ser `32` (o `out_channels` do primeiro). O `out_channels` será `64`. Siga este padrão para o terceiro bloco.

```

* **Classificador**:
* Comece definindo o contêiner sequencial: `self.classifier = nn.Sequential(...)`
**1. Camada Flatten**: A primeira camada é `nn.Flatten()`. Ela não recebe argumentos.
**2. Primeira Camada Linear**: Esta é `nn.Linear(in_features=..., out_features=512)`.
* Para encontrar o `in_features`, você precisa calcular o tamanho do tensor achatado (*flattened*). As imagens de entrada são 32x32. Cada `CNNBlock` contém uma camada `MaxPool2d` com um *stride* de 2, que reduz a altura e a largura pela metade. Após três blocos, as dimensões serão `32 → 16 → 8 → 4`.
* O último `CNNBlock` gera 128 canais na saída. Portanto, o número total de características é `128 * 4 * 4`.
**3. Camada ReLU**: Adicione `nn.ReLU()`.
**4. Camada de Dropout**: Adicione `nn.Dropout(0.6)`.
**5. Camada Linear Final**: Esta é `nn.Linear(in_features=512, out_features=num_classes)`.





**Para o método `forward**`:

* Este método descreve como os dados fluem da entrada para a saída. Você pode usar a mesma variável `x` e reatribuí-la após cada etapa.
* O pseudocódigo para a sequência é:
* `x = passe a entrada x através de self.conv_block1`
* `x = passe o novo x através de self.conv_block2`
* `x = passe o novo x através de self.conv_block3`
* `x = passe o mapa de características final através de self.classifier`


* Por fim, retorne `x`.

</details>

In [ ]:
# CLASSE AVALIADA: SimpleCNN

class SimpleCNN(nn.Module):
    """
    Define uma arquitetura CNN simples utilizando CNNBlocks modulares.

    Este modelo empilha três blocos convolucionais reutilizáveis seguidos por um
    classificador totalmente conectado (fully connected) para realizar a classificação de imagens.
    """
    def __init__(self, num_classes):
        """
        Inicializa as camadas do modelo SimpleCNN.

        Argumentos:
            num_classes (int): O número de classes de saída para o classificador.
        """
        # Inicializa a classe pai nn.Module.
        super(SimpleCNN, self).__init__()
        
        ### INICIE O SEU CÓDIGO AQUI ###

        # Define o primeiro bloco convolucional.
        self.conv_block1 = None
        # Define o segundo bloco convolucional.
        self.conv_block2 = None
        # Define o terceiro bloco convolucional.
        self.conv_block3 = None

        # Define o bloco classificador sequencial totalmente conectado.
        self.classifier = None(
            # Achata (flatten) o mapa de características 3D (canais, altura, largura) em um vetor 1D.
            None,
            # Primeira camada totalmente conectada (linear) que mapeia as características achatadas para uma camada oculta.
            None,
            # Função de ativação ReLU para introduzir não-linearidade.
            None,
            # Camada de Dropout para evitar sobreajuste (overfitting), definindo aleatoriamente uma fração das entradas para zero.
            None,
            # Camada totalmente conectada (linear) final que mapeia a camada oculta para as classes de saída.
            None
        ) 
        
        ### TÉRMINE O SEU CÓDIGO AQUI ###

    def forward(self, x):
        """
        Define a propagação para frente (forward pass) do modelo SimpleCNN.

        Argumentos:
            x (torch.Tensor): O tensor de entrada contendo um lote (batch) de imagens.

        Retorna:
            torch.Tensor: O tensor de saída com os logits para cada classe.
        """
        
        ### INICIE O SEU CÓDIGO AQUI ###
        
        # Passa a entrada através do primeiro bloco convolucional.
        x = None
        # Passa o resultado através do segundo bloco convolucional.
        x = None
        # Passa o resultado através do terceiro bloco convolucional.
        x = None

        # Passa o mapa de características final através do classificador.
        x = None
        
        ### TÉRMINE O SEU CÓDIGO AQUI ###
        
        # Retorna o tensor de saída final.
        return x

In [ ]:
# Verifica a SimpleCNN
print("--- Verificando SimpleCNN ---\n")

# Verifica a estrutura do modelo
# Instancia o modelo com 15 classes de saída (conforme o subconjunto do CIFAR-100)
verify_simple_cnn = SimpleCNN(num_classes=15)
print("Estrutura do Modelo (Model Structure):\n")
print(verify_simple_cnn)

# Verifica o formato (shape) da saída após uma propagação para frente (forward pass)
# Cria um tensor de entrada fictício (tamanho_do_lote=64, canais=3, altura=32, largura=32)
dummy_input = torch.randn(64, 3, 32, 32)
print(f"\nFormato do tensor de entrada: {dummy_input.shape}")

# Passa o tensor fictício através do modelo CNN criado
output = verify_simple_cnn(dummy_input)
print(f"Formato do tensor de saída:   {output.shape}")

#### Saída Esperada:

```
Estrutura do Modelo (Model Structure):

SimpleCNN(
  (conv_block1): CNNBlock(
    (block): Sequential(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU()
      (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
  )
  (conv_block2): CNNBlock(
    (block): Sequential(
      (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU()
      (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
  )
  (conv_block3): CNNBlock(
    (block): Sequential(
      (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU()
      (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=2048, out_features=512, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.6, inplace=False)
    (4): Linear(in_features=512, out_features=15, bias=True)
  )
)

Formato do tensor de entrada: torch.Size([64, 3, 32, 32])
Formato do tensor de saída: torch.Size([64, 15])
```

<br>

With your `SimpleCNN` class defined, the next step is to create an instance of the model.

* First, dynamically determine the number of classes by getting the length of the `.classes` attribute from your `train_dataset`.
* Next, create an instance of your `SimpleCNN` model, passing the `num_classes` variable to its constructor. This ensures the final layer of your model is correctly sized for your 15-class problem.

In [ ]:
# Obtém a quantidade total de classes presentes no conjunto de dados de treinamento
num_classes = len(train_dataset.classes)

# Instancia o modelo arquitetural utilizando o número de classes detectado
model = SimpleCNN(num_classes)

<a name='3'></a>
## 3 - Treinando o Modelo Aprimorado

Com o seu *pipeline* de dados aprimorado e a arquitetura CNN modular concluídos, você está pronto para iniciar o processo de treinamento. Nesta seção, você configurará as peças finais do seu *pipeline* de treinamento: a função de perda e o otimizador. Em seguida, você implementará a lógica central de treinamento e validação que executará o seu experimento e revelará o desempenho do seu novo modelo.

<a name='3-1'></a>
### 3.1 - Configurando a Função de Perda e o Otimizador

Antes que você possa treinar o modelo, você deve definir dois componentes principais: uma função de perda para medir o erro e um otimizador para atualizar os pesos do modelo.

* Para a função de perda, você continuará a usar o `nn.CrossEntropyLoss`, a escolha padrão para classificação multiclasse.

* Para o otimizador, você usará o `Adam`, mas com uma adição importante para combater o *overfitting*: `weight_decay`.
* O decaimento de peso (*weight decay*) adiciona uma penalidade à função de perda com base na magnitude dos pesos do modelo. Ele incentiva a rede a aprender valores de peso menores e mais simples, o que a torna mais robusta e menos propensa a memorizar os dados de treinamento. Esta é outra ferramenta vital para melhorar a capacidade de generalização do seu modelo.

In [ ]:
# Função de perda (Loss function)
# Utiliza a Entropia Cruzada, ideal para problemas de classificação multiclasse
loss_function = nn.CrossEntropyLoss()

# Otimizador para o modelo com decaimento de pesos (weight decay)
# Instancia o algoritmo Adam para atualizar os parâmetros do modelo com taxa de aprendizado e regularização L2
optimizer = optim.Adam(model.parameters(), lr=0.0005, weight_decay=0.0005)

<a name='3-2'></a>
### 3.2 - Implementando a Lógica de Treinamento e Validação

Você agora implementará a lógica central para treinar e avaliar o seu modelo. Isso será feito em duas funções separadas:

* `train_epoch`: Para realizar uma única passagem pelos dados de treinamento com o objetivo de atualizar o modelo.
* `validate_epoch`: Para realizar uma única passagem pelos dados de validação com o objetivo de medir o desempenho.

<a name='ex-4'></a>
### Exercício 4 - train_epoch

Sua tarefa é completar a lógica central de treinamento dentro do loop `for` da função `train_epoch`. Você implementará as cinco etapas fundamentais de uma única iteração de treinamento.

**Sua Tarefa**:

Dentro da função `train_epoch`, para cada lote de `images` e `labels`:

* **Zerar os Gradientes**:
* Antes de calcular os gradientes para o lote atual, você deve limpar (zerar) quaisquer gradientes que foram armazenados do lote anterior.


* **Passagem Direta (*Forward Pass*)**:
* Alimente o `model` com as `images` para obter as previsões de saída.


* **Calcular a Perda**:
* Use a `loss_function` fornecida para medir a diferença entre as saídas (`outputs`) do modelo e os rótulos verdadeiros (`labels`).


* **Passagem Reversa (*Backward Pass*)**:
* Calcule os gradientes da perda em relação a todos os parâmetros do modelo. Isso também é conhecido como retropropagação (*backpropagation*).


* **Atualizar os Parâmetros**:
* Use o `optimizer` para ajustar os parâmetros do modelo com base nos gradientes que você acabou de calcular.

<details>
<summary><b><font color="green">Dicas Adicionais de Código (Clique para expandir se estiver com dificuldades)</font></b></summary>

Se você precisar de ajuda, aqui está um guia mais direto para cada etapa.

* **Zerar os Gradientes**: Isso é feito para evitar o acúmulo de gradientes entre os lotes (*batches*).
* O pseudocódigo é: `chame o método zero_grad() do optimizer`.

* **Passagem Direta (*Forward Pass*)**: É assim que você obtém as previsões do modelo para o lote atual.
* O pseudocódigo é: `outputs = chame o model, passando as images como argumento`.

* **Calcular a Perda**: Você compara as previsões do modelo com os rótulos verdadeiros reais (*ground truth*).
* O pseudocódigo é: `loss = chame a loss_function, passando outputs e labels como argumentos`.

* **Passagem Reversa (*Backward Pass*)**: Esta etapa calcula o quanto cada parâmetro do modelo contribuiu para a perda geral.
* O pseudocódigo é: `chame o método backward() do tensor loss`.

* **Atualizar os Parâmetros**: O otimizador usa os gradientes calculados para dar um pequeno passo na direção que minimiza a perda.
* O pseudocódigo é: `chame o método step() do optimizer`.

</details>

In [ ]:
# FUNÇÃO AVALIADA: train_epoch

def train_epoch(model, train_loader, loss_function, optimizer, device):
    """
    Realiza uma única época de treinamento.

    Argumentos:
        model (torch.nn.Module): O modelo de rede neural a ser treinado.
        train_loader (torch.utils.data.DataLoader): O DataLoader para os dados de treinamento.
        loss_function (chamável): A função de perda.
        optimizer (torch.optim.Optimizer): O otimizador.
        device (torch.device): O dispositivo (CPU ou GPU) onde o treinamento será executado.

    Retorna:
        float: A perda (loss) média de treinamento para esta época.
    """
    # Define o modelo para o modo de treinamento
    model.train()
    running_loss = 0.0
    
    # Itera sobre os lotes (batches) de dados no carregador de treinamento
    for images, labels in train_loader:
        # Move as imagens e os rótulos para o dispositivo especificado (GPU ou CPU)
        images, labels = images.to(device), labels.to(device)
        
        ### INICIE O SEU CÓDIGO AQUI ###
        
        # Limpa os gradientes de todas as variáveis otimizadas
        None
        # Realiza a propagação para frente (forward pass) para obter as saídas do modelo
        outputs = None
        # Calcula a perda (loss)
        loss = None
        # Realiza a propagação para trás (backward pass) para computar os gradientes
        None
        # Atualiza os parâmetros do modelo
        None
        
        ### TÉRMINE O SEU CÓDIGO AQUI ###
        
        # Acumula a perda de treinamento para o lote atual
        running_loss += loss.item() * images.size(0)
        
    # Calcula e retorna a perda média de treinamento para a época
    epoch_loss = running_loss / len(train_loader.dataset)
    return epoch_loss

In [ ]:
# Utiliza uma função utilitária para realizar um teste de sanidade na implementação de train_epoch
helper_utils.verify_training_process(SimpleCNN, train_loader, loss_function, train_epoch, device)

#### Saída Esperada (Aproximadamente):

```
Treinando em 640 imagens por 5 épocas:

Época [1/5], Loss: 2.6735
Época [2/5], Loss: 2.3238
Época [3/5], Loss: 2.0528
Época [4/5], Loss: 1.8341
Época [5/5], Loss: 1.7676

Verificação da Atualização de Pesos: Os pesos do modelo mudaram durante o treinamento.
Verificação da Tendência de Perda: A perda diminuiu de 2.6735 para 1.7676.
```

<a name='ex-5'></a>
Você tem toda a razão, peço desculpas pela desatenção. Eu acabei convertendo a formatação e removendo os marcadores `>` na tentativa anterior.

Aqui está a tradução exata, mantendo rigorosamente todos os marcadores de recuo originais:

### Exercício 5 - validate_epoch

Sua tarefa é completar a lógica de validação. Isso envolve realizar uma passagem direta e, em seguida, calcular tanto a perda quanto o número de previsões corretas para determinar a acurácia.

**Sua Tarefa**:

* **Desabilitar o Cálculo de Gradientes**:

> * Envolva todo o loop for dentro do gerenciador de contexto `torch.no_grad()`. Isso diz ao PyTorch para não calcular gradientes, o que economiza memória e tempo de computação durante a validação.

* **Dentro do loop `for`**:

>* **Passagem Direta**: 
>>* Assim como no treinamento, passe as `images` através do `model` para obter as suas `outputs`.        
> * **Calcular a Perda**: 
>>* Use a `loss_function` para calcular a `val_loss` entre as `outputs` e os `labels` verdadeiros.
> * **Acumular a Perda**: 
>>* Adicione a perda do lote à `running_val_loss`. Lembre-se de obter o valor escalar do tensor de perda e multiplicá-lo pelo tamanho do lote.
> * **Obter as Previsões**: 
>>* Determine a classe prevista pelo modelo para cada imagem no lote. As `outputs` do seu modelo são pontuações brutas (logits). A classe com a pontuação mais alta é a previsão do modelo. Você precisa encontrar o índice dessa pontuação máxima.





<details>
<summary><b><font color="green">Dicas Adicionais de Código (Clique para expandir se estiver com dificuldades)</font></b></summary>

Se você precisar de um pouco mais de direção, aqui está um guia detalhado.

**Desabilitar o Cálculo de Gradientes**:

Este é um gerenciador de contexto no PyTorch. A estrutura que você precisa é `with torch.no_grad():`. O loop `for` deve ser recuado (indentado) dentro deste bloco.

**Dentro do loop `for**`:

* **Passagem Direta (*Forward Pass*)**: Isso é idêntico ao loop de treinamento. O pseudocódigo é: `outputs = chame o model, passando as images como argumento`.
* **Calcular a Perda**: Isso também é o mesmo que no loop de treinamento. O pseudocódigo é: `val_loss = chame a loss_function, passando outputs e labels como argumentos`.
* **Acumular a Perda**: Você precisa atualizar a variável `running_val_loss`. O pseudocódigo é: `running_val_loss += obtenha o valor escalar de val_loss usando o método .item() * o número de imagens no lote atual`.
* **Obter as Previsões**: Você precisa encontrar a classe mais provável a partir dos *logits* de saída.
* A função `torch.max()` é perfeita para isso. Você precisa chamá-la no tensor `outputs` ao longo da dimensão 1 (a dimensão das classes).
* O pseudocódigo é: `_, predicted = use torch.max() no tensor outputs, especificando a dimensão 1`.
* Note que `torch.max()` retorna uma tupla de `(max_values, max_indices)`. Você precisa apenas do segundo elemento, os índices, que correspondem aos rótulos das classes previstas.

</details>

In [ ]:
# FUNÇÃO AVALIADA: validate_epoch

def validate_epoch(model, val_loader, loss_function, device):
    """
    Realiza uma única época de validação.

    Argumentos:
        model (torch.nn.Module): O modelo de rede neural a ser validado.
        val_loader (torch.utils.data.DataLoader): O DataLoader para os dados de validação.
        loss_function (chamável): A função de perda.
        device (torch.device): O dispositivo (CPU ou GPU) onde a validação será executada.

    Retorna:
        tuple: Uma tupla contendo a perda (loss) média de validação e a acurácia de validação.
    """
    # Define o modelo para o modo de avaliação
    model.eval()
    running_val_loss = 0.0
    correct = 0
    total = 0
    
    ### INICIE O SEU CÓDIGO AQUI ###
    
    # Desativa o cálculo de gradientes para a validação (economiza memória e processamento)
    with None:
        
    ### TÉRMINE O SEU CÓDIGO AQUI ###
        
        # Itera sobre os lotes (batches) de dados no carregador de validação
        for images, labels in val_loader:
            # Move as imagens e os rótulos para o dispositivo especificado (GPU ou CPU)
            images, labels = images.to(device), labels.to(device)
            
            ### INICIE O SEU CÓDIGO AQUI ###
            
            # Realiza a propagação para frente (forward pass) para obter as saídas do modelo
            outputs = None
            
            # Calcula a perda (loss) de validação para o lote atual
            val_loss = None
            # Acumula a perda de validação
            running_val_loss += None
            
            # Obtém os índices/rótulos das classes preditas (maior logit)
            _, predicted = None
            
            ### TÉRMINE O SEU CÓDIGO AQUI ###
            
            # Atualiza o número total de amostras processadas
            total += labels.size(0)
            # Atualiza o número de predições corretas
            correct += (predicted == labels).sum().item()
            
    # Calcula a perda média de validação e a acurácia para esta época
    epoch_val_loss = running_val_loss / len(val_loader.dataset)
    epoch_accuracy = 100.0 * correct / total
    
    return epoch_val_loss, epoch_accuracy

In [ ]:
# Utiliza uma função utilitária para realizar um teste de sanidade na implementação de validate_epoch
helper_utils.verify_validation_process(SimpleCNN, val_loader, loss_function, validate_epoch, device)

#### Saída Esperada:

```
Verificação de Tipos de Retorno: A função retornou um float para perda e acurácia.
Verificação de Integridade dos Pesos: Os pesos do modelo não foram alterados durante a validação.
```

Com as funções individuais de treinamento e validação concluídas, você pode agora reuni-las no `training_loop` principal. Esta função orquestra todo o processo de treinamento ao longo de um número definido de épocas e inclui uma atualização fundamental.

Um desafio comum é que o desempenho de um modelo pode atingir o pico e, em seguida, declinar se o treinamento continuar por muito tempo. Para resolver isso, o `training_loop` irá:

* **Monitorar** a acurácia de validação ao final de cada época.
* **Manter o registro** do estado do modelo com o melhor desempenho visto até o momento.
* Após a época final, ele automaticamente **retorna o modelo da sua única melhor época**.

Isso garante que você sempre obtenha de volta a versão do seu modelo que alcançou a maior acurácia de validação durante toda a execução do treinamento.

In [ ]:
def training_loop(model, train_loader, val_loader, loss_function, optimizer, num_epochs, device):
    """
    Treina e valida un modelo de rede neural no PyTorch.

    Argumentos:
        model (torch.nn.Module): O modelo a ser treinado.
        train_loader (torch.utils.data.DataLoader): DataLoader para o conjunto de treinamento.
        val_loader (torch.utils.data.DataLoader): DataLoader para o conjunto de validação.
        loss_function (chamável): A função de perda (loss).
        optimizer (torch.optim.Optimizer): O algoritmo de otimização.
        num_epochs (int): O número total de épocas para realizar o treinamento.
        device (torch.device): O dispositivo (ex: 'cuda' ou 'cpu') onde o treinamento será executado.

    Retorna:
        tuple: Uma tupla contendo o melhor modelo treinado e uma lista com as métricas históricas
               (train_losses, val_losses, val_accuracies).
    """
    # Move o modelo para o dispositivo especificado (CPU ou GPU)
    model.to(device)
    
    # Inicializa variáveis para monitorar e salvar o modelo com melhor desempenho
    best_val_accuracy = 0.0
    best_model_state = None
    best_epoch = 0
    
    # Inicializa as listas para armazenar as métricas de treinamento e validação por época
    train_losses, val_losses, val_accuracies = [], [], []
    
    print("--- Treinamento Iniciado ---")
    
    # Itera ao longo do número especificado de épocas
    for epoch in range(num_epochs):
        # Realiza uma época completa de treinamento
        epoch_loss = train_epoch(model, train_loader, loss_function, optimizer, device)
        train_losses.append(epoch_loss)
        
        # Realiza uma época completa de validação
        epoch_val_loss, epoch_accuracy = validate_epoch(model, val_loader, loss_function, device)
        val_losses.append(epoch_val_loss)
        val_accuracies.append(epoch_accuracy)
        
        # Exibe as métricas de desempenho da época atual
        print(f"Época [{epoch+1}/{num_epochs}], Perda Treino: {epoch_loss:.4f}, Perda Val: {epoch_val_loss:.4f}, Acurácia Val: {epoch_accuracy:.2f}%")
        
        # Verifica se o modelo atual superou a melhor acurácia de validação obtida até então
        if epoch_accuracy > best_val_accuracy:
            best_val_accuracy = epoch_accuracy
            best_epoch = epoch + 1
            # Cria uma cópia profunda (deep copy) do estado dos pesos do melhor modelo em memória
            best_model_state = copy.deepcopy(model.state_dict())
            
    print("--- Treinamento Finalizado ---")
    
    # Carrega os pesos do melhor modelo restaurado antes de retornar
    if best_model_state:
        print(f"\n--- Retornando o melhor modelo com {best_val_accuracy:.2f}% de acurácia de validação, obtido na época {best_epoch} ---")
        model.load_state_dict(best_model_state)
    
    # Consolida todos os históricos de métricas em uma única lista
    metrics = [train_losses, val_losses, val_accuracies]
    
    # Retorna o modelo otimizado e as respectivas métricas coletadas
    return model, metrics

Tudo está pronto agora. O código a seguir chamará a sua função `training_loop` para iniciar o processo completo de treinamento e validação.

O modelo treinará por **50 épocas**. Com as poderosas técnicas de regularização que você adicionou (*Batch Normalization*, *Dropout* aumentado e *Weight Decay*) e uma taxa de aprendizado menor, o modelo é projetado para aprender de forma mais cautelosa. Esta execução de treinamento mais longa dá ao modelo tempo suficiente para convergir para uma solução robusta e generalizada.

In [ ]:
# Inicia o processo de treinamento chamando a função do loop principal (training loop)
trained_model, training_metrics = training_loop(
    model=model, 
    train_loader=train_loader, 
    val_loader=val_loader, 
    loss_function=loss_function, 
    optimizer=optimizer, 
    num_epochs=50, 
    device=device
)

# Visualiza as métricas de desempenho obtidas (gráficos de perda e acurácia)
print("\n--- Gráficos do Treinamento ---\n")
helper_utils.plot_training_metrics(training_metrics)

**Analisando os Resultados**

Observe atentamente os novos gráficos de treinamento e compare-os com os do laboratório anterior. A diferença é notável.

As curvas de perda de treinamento e validação agora se acompanham de perto, e a grande lacuna que sinalizava o *overfitting* desapareceu. A acurácia de validação mostra uma subida muito mais saudável e consistente. Esta é uma evidência clara de que você resolveu com sucesso o problema de *overfitting*! A combinação de mais aumento de dados (*data augmentation*), Normalização em Lote (*Batch Normalization*) e Decaimento de Peso (*Weight Decay*) trabalhou em conjunto para criar um modelo que generaliza muito melhor do que antes.

**O Platô de Desempenho**

A acurácia de validação do seu modelo agora atinge um pico em torno de 70%, o que é um resultado sólido. Você pode se perguntar, no entanto, por que não atingiu 90% ou mais, especialmente com todas essas técnicas avançadas e um treinamento mais longo. A resposta está na eficácia com que você utilizou as ferramentas à sua disposição.

Os fundamentos que você aprendeu neste curso fornecem uma base sólida para a construção de modelos de *deep learning*. As técnicas agora à sua disposição, desde o aumento de dados até o design modular e a regularização, são poderosas. Aplicá-las corretamente é precisamente o que permitiu que você resolvesse o problema inicial de *overfitting* e alcançasse esse forte resultado. Isso demonstra que você está expandindo os limites do que pode ser alcançado com este conjunto de ferramentas fundamentais.

Você alcançou algo significativo. Você começou construindo uma CNN simples que sofria de um problema comum e desafiador, e sistematicamente aprimorou todo o seu *pipeline* com técnicas profissionais para criar este modelo final e robusto. Parabéns pelo resultado bem-sucedido!

<a name='4'></a>
## 4 - Além dos Fundamentos: Um Vislumbre do Próximo Nível

Você transformou com sucesso uma CNN simples, diagnosticou suas falhas e a aprimorou sistematicamente em um modelo robusto e bem generalizado. Você expandiu o conjunto de ferramentas fundamentais que aprendeu até os seus limites para alcançar um resultado forte.

**Mas e se este não for o limite? E se houvesse outra maneira?**

**E se você pudesse levar a acurácia do seu modelo de cerca de 70% para mais de 80% neste exato mesmo conjunto de dados?**

Dê uma olhada nos resultados de uma estratégia de treinamento diferente e mais poderosa. Execute a próxima célula para ver isso na prática.

In [ ]:
# Importa a função de pré-visualização que demonstra conceitos do próximo curso
from c2_preview.c2_preview import course_2_preview

# Esta função utilitária executa um loop de treinamento usando uma estratégia avançada
# que será ensinada em curso futuro. Execute esta célula para ver os resultados aprimorados em ação.
trained_model = course_2_preview(
    train_dataset, 
    val_dataset, 
    loss_function,
    device,
    num_epochs=5
)

Incrível, não é? Em apenas **5 épocas**, a acurácia de validação ultrapassou os 80%, um nível de desempenho que o seu modelo anterior não alcançou mesmo após 50 épocas.

**Como é possível uma melhoria tão rápida e dramática exatamente nos mesmos dados?**

Este resultado foi alcançado combinando várias técnicas poderosas de próximo nível que você dominará no próximo curso. Esta foi apenas uma prévia, mas a estratégia envolveu três atualizações principais:

* **Usando um Modelo Pré-treinado**: Esta é a mudança mais significativa. Em vez de começar do zero com pesos aleatórios, esta abordagem usa um modelo sofisticado que já foi treinado em milhões de imagens. Ele já possui uma compreensão profunda de padrões visuais, que você pode então ajustar (*fine-tune*) para a sua tarefa específica.
* **Escalonamento Dinâmico da Taxa de Aprendizado**: Em vez de usar uma taxa de aprendizado única e fixa, esta estratégia usa um *escalonador de taxa de aprendizado* (*learning rate scheduler*). Esta ferramenta ajusta inteligentemente a taxa de aprendizado durante o treinamento, fazendo atualizações maiores no início e ajustes menores e mais precisos à medida que o modelo se aproxima da melhor solução.
* **Transformações Mais Avançadas**: O *pipeline* de aumento de dados usado para esta prévia também era mais avançado. Ele incluiu técnicas adaptadas especificamente para esses modelos de alto desempenho, garantindo que a rede aprendesse a partir de um conjunto mais rico e desafiador de exemplos de treinamento.

Esses conceitos são apenas um vislumbre do que vem a seguir. Você construiu uma base incrível e agora está pronto para aprender as estratégias que os profissionais usam para alcançar resultados de ponta (*state of the art*) de forma rápida e eficiente.

## Conclusão

Parabéns por concluir esta tarefa!

Você navegou com sucesso por um fluxo de trabalho de *machine learning* completo e realista. Você começou com um modelo que sofria de *overfitting*, diagnosticou o problema e, em seguida, aplicou sistematicamente uma série de técnicas profissionais e poderosas para resolvê-lo. Você não apenas melhorou um modelo; você aprendeu um processo repetível para refinar e fortalecer qualquer rede neural que construir no futuro.

As habilidades que você praticou aqui, design modular, implementação de regularização e análise da dinâmica de treinamento, são fundamentais para construir modelos eficazes de *deep learning*. Você foi além do básico e agora está equipado com o conhecimento prático necessário para enfrentar problemas mais complexos do mundo real. Muito bem!